> 📝 **Planning note (author use -- remove before publishing)**
>
> **Intended contents:** `.hex`/`.XMLCON` -> `.cnv` conversion, load a `.cnv`, first look at data + metadata, drop unused variables, save to NetCDF.
>
> **To do:**
> - ~~Add a planning note~~ (done).
> - Consider whether the `.cnv` conversion section needs a screenshot or is fine as text.

<img src="../../data/images/hiaoos_learning_moored.png" width="300" align="right">

# Load data - SBE37 instrument


In this notebook, we will:

- Explain how to **convert raw data** (`.hex`., `.XMLCON`) **to a physically readable data format** (`.cnv`).   
- **Load data** from a raw `.cnv` file obtained from an **Seabird SBE37** CTD.
- Have a **quick look** at the data and metadata.
- **Store the data as a NetCDF file** for subsequent use.

*Notebook 3 of 10 in the series*  
[← 2. Loading data (RBR)](./02_load_data_rbr_instruments.ipynb) · [4. Post-processing & QC I →](./04_post_proc_qc_deck_time_outliers.ipynb)

___

#### Imports 
We start with importing `kval.moored` which contains some helper functions for loading `.cnv` files.
- Data and metadata (including information about the instrument, serial number, etc) are read from the input file.
- Data and metadata are organized into our preferred format (`xarray` *Dataset*, which makes for easy data workflow, and excellent compatibility with netCDF format).

In [21]:
from kval.data import moored
%matplotlib widget

___

## NOTE: `.cnv` files

This notebook starts from human-readable `.cnv` files, where raw data have been converted to physical values through existing calibration coefficients. `.cnv` files can be produced when reading data from the instrument using Seabird's *Seaterm* acquisition software (go to `Tools - Convert .XML data file`). 

In some cases, you may only have raw files (`.hex`., `.XMLCON`) available - in this case, you can convert to `.cnv` using the *SBE Data Processing* software [<sup id="fnref1"><a href="#fn1">1</a></sup>].

To convert files using SBE Data Processing, use `Run -> Data Conversion`

> <sup id="fn1">1</sup>: *A newer alternative to SBEProcessing, *Fathom*, is now available. So too is a Python package from SeaBird, [seabirdscientific](https://github.com/Sea-BirdScientific/seabirdscientific). `kval` intends to use functionality from this package eventually, but this has not been implemented yet - feel free to experiment (but note that there does not seem to be an easy way to digest `.XMLCON` files at this point). <a href="#fnref1">↩</a>


---





## Load data

Specify the file you want to load 

In [22]:
sbe37_file = '../../data/moored_CTD_test_data/raw_data/AT200_21_22_SBE37_20773_49m.cnv'

.. and load it into an xarray Dataset, `ds`:

In [23]:
ds = moored.load_moored(sbe37_file, lat = 81.4105, lon = 31.2433)

It is good to specify the latitude and longitude here. They are useful in most applications, and are also used directly in some subsequent processing steps. 

Running `ds` in a cell will now display an overview of the contents of the file. 

Note:
- Some known variables will have been renamed to canonical variable names like (`TEMP`, `PRES`)
- The lat/lon we assigned can be found as coordinate variables `LATITUDE`, `LONGITUDE`
- Where available, serial numbers and calibration dates can be found in the metadata attributes of each variables.
- Some overarching metadata can be found in the general attributes.  

In [24]:
ds

<xarray.Dataset> Size: 2MB
Dimensions:    (TIME: 31951)
Coordinates:
  * TIME       (TIME) float64 256kB 1.894e+04 1.894e+04 ... 1.927e+04 1.927e+04
    LATITUDE   float32 4B 81.41
    LONGITUDE  float32 4B 31.24
Data variables:
    CNDC       (TIME) float64 256kB 0.000367 0.000368 0.000368 ... 18.13 18.24
    PRES       (TIME) float64 256kB -0.101 -0.101 -0.108 ... -0.287 -0.289
    PSAL       (TIME) float64 256kB 0.0015 0.0016 0.0016 ... 15.55 15.53 15.55
    TEMP       (TIME) float64 256kB 4.377 4.478 4.436 ... 9.653 9.879 10.03
    timeM      (TIME) float64 256kB 15.0 30.0 45.0 ... 4.792e+05 4.793e+05
    timeH      (TIME) float64 256kB 0.25 0.5 0.75 ... 7.988e+03 7.988e+03
    TIME_JULD  (TIME) float64 256kB 312.0 312.0 312.0 ... 644.8 644.8 644.8
    SBE_FLAG   (TIME) float64 256kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0
Attributes:
    source_file:               sbe37smp-rs232_03720773_2022_10_06.hex, SBE37S...
    history:                   2021-11-07 - 2022-10-06: Data collection.\n202...
    instrument_model:          Sea-Bird SBE37SMP-RS232
    SBE_processing:            SBE SOFTWARE PROCESSING STEPS (extracted from ...
    SBE_processing_date:       2022-10-06T19:09:06Z
    SBE_flags_applied:         yes
    instrument_serial_number:  20773
    time_coverage_resolution:  P0000-00-00T00:15:00

___

## Have a quick look at the data

We now have the data imported in to an xarray Dataset, with variables like `TIME`, `CNDC`, `TEMP`, `PRES`. We have also collected some useful metadata such as serial numbers, calibration dates, etc.

First, have a look at the contents of the data by executing `ds` (next cell). This will display an overview of the dataset, including dimensions (in this case, 31953 points in the `TIME` dimension), variables, and metadata. 

Click around to get a sense of the contents of the dataset: 
- Click 📄 next to a variable to see its metadata
- Click ⛃ next to a variable to see a subset of the data.
- Global metadata attributes are found near the bottom, under *Attributes*

> **Note: why does `TIME` look like a number?**
>
> As in [notebook 2](./02_load_data_rbr_instruments.ipynb): `TIME` is stored the standard NetCDF/CF way, as a number of units counted from a fixed reference (here *days since 1970-01-01*, recorded in the variable's `units` attribute) rather than as a date. We return to converting between the two in [notebook 6](./06_slicing_averaging_interpolating_correlating.ipynb).

___

> **Note: Variables in the file**
> Depending on how the file was exported, your `.cnv` may contain different parameters; including derived variables like `PSAL` and other variables like `timeM`.
>
> The only variables that are necessary are `TIME` (which was parsed from one of the other time fields), and the variables the instrument, in this case: `PRES`, `TEMP`, and `CNDC`
>
> You can retain any variables you'd like, including salinity `PSAL`. We can also easily recompute it later.
>
____

In the following line we will remove unused variables: 

In [25]:
ds = ds[['PRES', 'CNDC', 'TEMP', 'PSAL']]

If we look at the updated dataset, it should only contain the core variables. 

In [26]:
ds

<xarray.Dataset> Size: 1MB
Dimensions:    (TIME: 31951)
Coordinates:
  * TIME       (TIME) float64 256kB 1.894e+04 1.894e+04 ... 1.927e+04 1.927e+04
    LATITUDE   float32 4B 81.41
    LONGITUDE  float32 4B 31.24
Data variables:
    PRES       (TIME) float64 256kB -0.101 -0.101 -0.108 ... -0.287 -0.289
    CNDC       (TIME) float64 256kB 0.000367 0.000368 0.000368 ... 18.13 18.24
    TEMP       (TIME) float64 256kB 4.377 4.478 4.436 ... 9.653 9.879 10.03
    PSAL       (TIME) float64 256kB 0.0015 0.0016 0.0016 ... 15.55 15.53 15.55
Attributes:
    source_file:               sbe37smp-rs232_03720773_2022_10_06.hex, SBE37S...
    history:                   2021-11-07 - 2022-10-06: Data collection.\n202...
    instrument_model:          Sea-Bird SBE37SMP-RS232
    SBE_processing:            SBE SOFTWARE PROCESSING STEPS (extracted from ...
    SBE_processing_date:       2022-10-06T19:09:06Z
    SBE_flags_applied:         yes
    instrument_serial_number:  20773
    time_coverage_resolution:  P0000-00-00T00:15:00

___

### Quick look continued
We now have the data on our preferred format, and can make some quick plots using `xarray`..

In [27]:
ds.TEMP.plot()

.. or use functionality from `kval` which lets us flip through variables and show e.g. temporal averages:

In [28]:
moored.plot(ds)

____

***So -*** we have now read the data with some basic metadata and gotten a basic dea of what it contains. We can now export directly to a NetCDF file. 

In subsequent notebooks, we will load the file we create here and go on to do some post-processing and analysis, and we will save new updated versions of the data as we go.
___

## Save to NetCDF

In [29]:
# Specify where we want to save the file to
out_path = '../../data/moored_CTD_test_data/intermediate_data/'

# Specify a file name
out_name = 'AT200_21_22_SBE37_20773_49m_from_raw.nc'

In [30]:
moored.to_netcdf(ds, out_path, out_name)

Updated history attribute. Current content:
---
2021-11-07 - 2022-10-06: Data collection.
2022-10-06: Processed to .cnv using SBE software.
2026-09-25: Post-processing.
2026-09-25: Creation of this netcdf file.
---
Exported NetCDF file as: ../../data/moored_CTD_test_data/intermediate_data/AT200_21_22_SBE37_20773_49m_from_raw.nc


___

**All done!** Hopefully this produced a file - check that it was created where you expected it to.
___

Note that you can also export to other formats, e.g. a MATLAB data file:

    moored.to_mat(ds, 'path/dataset_as_mat.mat')

Here, we will stick with NetCDF.

___

**Next:** [4. Post-processing & QC I →](./04_post_proc_qc_deck_time_outliers.ipynb)